# COCO-small MVP Pipeline (Colab)

Этот ноутбук повторяет структуру `mvp_pipeline_colab.ipynb`,
но для датасета COCO 2017 с фокусом на малых объектах (`area <= 32^2`).


In [ ]:
# Step 0: install dependencies in Colab runtime.
# Если какие-то пакеты уже установлены, pip просто пропустит их.
%pip install -q ultralytics pycocotools albumentations pyyaml tqdm


In [ ]:
# Step 1: mount Google Drive and define dataset/experiment paths.
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

DRIVE_ROOT = Path('/content/drive/MyDrive')
COCO_ZIPS_DIR = DRIVE_ROOT / 'datasets' / 'coco2017_zips'
COCO_RAW_DIR = DRIVE_ROOT / 'datasets' / 'coco2017_raw'
COCO_SMALL_YOLO_DIR = DRIVE_ROOT / 'datasets' / 'coco_small_yolo'

# All pipeline artifacts for this dataset will be mirrored here.
COCO_EXPERIMENT_ROOT = DRIVE_ROOT / 'experiments' / 'small_objects_auto_aug' / 'coco_small'
COCO_EXPERIMENT_RUNS = COCO_EXPERIMENT_ROOT / 'runs'
COCO_EXPERIMENT_ARTIFACTS = COCO_EXPERIMENT_ROOT / 'artifacts'

COCO_ZIPS_DIR.mkdir(parents=True, exist_ok=True)
COCO_RAW_DIR.mkdir(parents=True, exist_ok=True)
COCO_SMALL_YOLO_DIR.mkdir(parents=True, exist_ok=True)
COCO_EXPERIMENT_RUNS.mkdir(parents=True, exist_ok=True)
COCO_EXPERIMENT_ARTIFACTS.mkdir(parents=True, exist_ok=True)

print('COCO zips dir:', COCO_ZIPS_DIR)
print('COCO raw dir:', COCO_RAW_DIR)
print('COCO small yolo dir:', COCO_SMALL_YOLO_DIR)
print('COCO experiment root:', COCO_EXPERIMENT_ROOT)


In [ ]:
# Step 2: clone/open project repository in runtime.
from pathlib import Path
import subprocess
import sys

PROJECT_ROOT = Path('/content/small_objects_auto_aug')
REPO_URL = 'https://github.com/s44w/small_objects_auto_aug.git'

if not PROJECT_ROOT.exists():
    subprocess.check_call(['git', 'clone', REPO_URL, str(PROJECT_ROOT)])

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Using PROJECT_ROOT:', PROJECT_ROOT)


In [ ]:
# Step 3: download official COCO 2017 archives to Google Drive (idempotent).
# If files already exist in Drive, download is skipped.
from src.data.coco_small_manager import COCO_ARCHIVES_2017, download_coco_2017_archives

FORCE_DOWNLOAD_COCO = False

existing_archives = {
    name: (COCO_ZIPS_DIR / name)
    for name in COCO_ARCHIVES_2017.keys()
}
missing_archives = [
    name for name, path in existing_archives.items()
    if not (path.exists() and path.stat().st_size > 0)
]

if missing_archives or FORCE_DOWNLOAD_COCO:
    print('[INFO] Downloading archives:', ', '.join(missing_archives) if missing_archives else 'all (forced)')
    archive_paths = download_coco_2017_archives(COCO_ZIPS_DIR)
else:
    print('[SKIP] All COCO archives already exist in Drive. Download step skipped.')
    archive_paths = existing_archives

for name, path in archive_paths.items():
    size_mb = path.stat().st_size / (1024 * 1024) if path.exists() else 0
    print(f'[OK] {name}: {path} ({size_mb:.1f} MB)')


In [ ]:
# Step 4: extract COCO archives in Drive (idempotent).
# If raw folders/files already exist, extraction is skipped.
from src.data.coco_small_manager import extract_coco_2017_archives

FORCE_EXTRACT_COCO = False

raw_markers = [
    COCO_RAW_DIR / 'train2017',
    COCO_RAW_DIR / 'val2017',
    COCO_RAW_DIR / 'annotations' / 'instances_train2017.json',
    COCO_RAW_DIR / 'annotations' / 'instances_val2017.json',
]
raw_ready = all(path.exists() for path in raw_markers)

if raw_ready and not FORCE_EXTRACT_COCO:
    print('[SKIP] COCO raw structure already exists in Drive. Extraction step skipped.')
    extracted = {
        'train2017': COCO_RAW_DIR / 'train2017',
        'val2017': COCO_RAW_DIR / 'val2017',
        'annotations': COCO_RAW_DIR / 'annotations',
    }
else:
    extracted = extract_coco_2017_archives(
        COCO_ZIPS_DIR,
        COCO_RAW_DIR,
        overwrite=bool(FORCE_EXTRACT_COCO),
    )

for name, path in extracted.items():
    print(f'[OK] extracted {name}: {path}')


In [ ]:
# Step 5: convert COCO raw -> YOLO (small objects only) (idempotent).
# If COCO-small YOLO dataset already exists, conversion is skipped.
from src.data.coco_small_manager import CocoSmallPrepareConfig, prepare_coco_small_yolo

FORCE_REBUILD_COCO_SMALL_YOLO = False

prepare_cfg = CocoSmallPrepareConfig(
    small_max_area=32.0**2,
    keep_images_without_small=False,
    include_crowd=False,
    splits=('train', 'val'),
    link_images=True,  # save Drive space by linking to raw images when possible
)

yolo_markers = [
    COCO_SMALL_YOLO_DIR / 'images' / 'train',
    COCO_SMALL_YOLO_DIR / 'images' / 'val',
    COCO_SMALL_YOLO_DIR / 'labels' / 'train',
    COCO_SMALL_YOLO_DIR / 'labels' / 'val',
    COCO_SMALL_YOLO_DIR / 'coco_small.yaml',
]

def _count_images(images_dir):
    exts = {'.jpg', '.jpeg', '.png', '.bmp'}
    if not images_dir.exists():
        return 0
    return sum(1 for p in images_dir.iterdir() if p.is_file() and p.suffix.lower() in exts)

yolo_ready = all(p.exists() for p in yolo_markers)
train_count = _count_images(COCO_SMALL_YOLO_DIR / 'images' / 'train')
val_count = _count_images(COCO_SMALL_YOLO_DIR / 'images' / 'val')
yolo_ready = yolo_ready and (train_count > 0 and val_count > 0)

if yolo_ready and not FORCE_REBUILD_COCO_SMALL_YOLO:
    print('[SKIP] COCO-small YOLO dataset already exists in Drive. Conversion step skipped.')
    print(f'[INFO] train images: {train_count}, val images: {val_count}')
else:
    prepare_report = prepare_coco_small_yolo(
        raw_coco_root=COCO_RAW_DIR,
        output_root=COCO_SMALL_YOLO_DIR,
        config=prepare_cfg,
    )
    print('is_valid:', prepare_report.is_valid)
    print('num_classes:', len(prepare_report.class_names))
    print('train stats:', prepare_report.splits.get('train', {}))
    print('val stats:', prepare_report.splits.get('val', {}))


In [ ]:
# Step 6: build runtime config for pipeline (based on configs/coco_small_config.yaml).
import yaml

base_cfg_path = PROJECT_ROOT / 'configs' / 'coco_small_config.yaml'
runtime_cfg_path = PROJECT_ROOT / 'artifacts' / 'coco_small_runtime_config.yaml'
runtime_cfg_path.parent.mkdir(parents=True, exist_ok=True)

with base_cfg_path.open('r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

cfg['dataset']['mode'] = 'manual'
cfg['dataset']['root'] = str(COCO_SMALL_YOLO_DIR)
cfg['dataset']['raw_root'] = str(COCO_RAW_DIR)
cfg['training']['data_yaml'] = str(COCO_SMALL_YOLO_DIR / 'coco_small.yaml')

# Persist heavy outputs directly to Google Drive.
cfg['training']['project_dir'] = str(COCO_EXPERIMENT_RUNS)
cfg['training']['run_ablation'] = False
cfg['training']['cache'] = 'disk'
cfg['training']['val_during_train'] = True

with runtime_cfg_path.open('w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, allow_unicode=True, sort_keys=False)

print('Runtime config:', runtime_cfg_path)
print('Training runs dir (Drive):', cfg['training']['project_dir'])


In [ ]:
# Step 7: smoke test (subset -> analyze -> policy).
from src.data.subset_builder import build_yolo_subset
from src.analysis.dataset_analyzer import DatasetAnalyzerConfig, analyze_dataset
from src.policy.rule_engine import RuleEngineConfig, generate_policy_from_stats, save_policy_artifacts
from src.utils.io import load_yaml

RUN_SMOKE = True

if RUN_SMOKE:
    cfg = load_yaml(runtime_cfg_path)

    subset_root = PROJECT_ROOT / 'datasets' / 'coco_small_smoke'
    smoke_stats_dir = PROJECT_ROOT / 'artifacts' / 'coco_small_smoke' / 'stats'
    smoke_policy_dir = PROJECT_ROOT / 'artifacts' / 'coco_small_smoke' / 'policy'

    subset_report = build_yolo_subset(
        dataset_root=COCO_SMALL_YOLO_DIR,
        output_root=subset_root,
        train_images=120,
        val_images=40,
        seed=42,
        clean_output=True,
    )
    print('subset report:', subset_report)

    stats = analyze_dataset(
        dataset_root=subset_root,
        output_dir=smoke_stats_dir,
        splits=('train',),
        config=DatasetAnalyzerConfig(
            small_max_area=float(cfg['analysis']['small_max_area']),
            medium_max_area=float(cfg['analysis']['medium_max_area']),
            tiny_max_area=float(cfg['analysis']['tiny_max_area']),
            image_extensions=tuple(cfg['dataset'].get('image_extensions', ['.jpg', '.jpeg', '.png', '.bmp'])),
            generate_plots=False,
        ),
    )

    policy, decision_report = generate_policy_from_stats(
        stats,
        cfg=RuleEngineConfig.from_project_config(cfg),
    )
    paths = save_policy_artifacts(policy, decision_report, output_dir=smoke_policy_dir)

    print('Smoke OK')
    print('stats:', smoke_stats_dir / 'dataset_stats.json')
    print('policy:', paths['policy_json'])
else:
    print('Smoke skipped')


In [ ]:
# Step 8: full pipeline (analysis + policy, optional training/eval) + Drive artifact sync.
from src.pipeline_mvp import run_mvp_pipeline
import shutil

RUN_TRAINING = False
RUN_EVAL = False
TRAIN_PROFILE = 'fast'  # fast | final | balanced | quality | hour | max_quality
SYNC_ARTIFACTS_TO_DRIVE = True

outputs = run_mvp_pipeline(
    project_config_path=runtime_cfg_path,
    run_training=RUN_TRAINING,
    run_eval=RUN_EVAL,
    train_profile=TRAIN_PROFILE,
    auto_predict_for_eval=True,
)
print(outputs)

if SYNC_ARTIFACTS_TO_DRIVE:
    local_artifacts = PROJECT_ROOT / 'artifacts'
    COCO_EXPERIMENT_ARTIFACTS.mkdir(parents=True, exist_ok=True)
    shutil.copytree(local_artifacts, COCO_EXPERIMENT_ARTIFACTS, dirs_exist_ok=True)
    print('[OK] Artifacts synced to Drive:', COCO_EXPERIMENT_ARTIFACTS)


In [ ]:
# Step 9 (optional): full training + eval in one call.
# ????????? ?????? ????? ?????? ????? ?????????? ??????.
from src.pipeline_mvp import run_mvp_pipeline
import shutil

RUN_FULL_TRAIN_EVAL = False
TRAIN_PROFILE = 'hour'
SYNC_ARTIFACTS_TO_DRIVE = True

if RUN_FULL_TRAIN_EVAL:
    outputs = run_mvp_pipeline(
        project_config_path=runtime_cfg_path,
        run_training=True,
        run_eval=True,
        train_profile=TRAIN_PROFILE,
        auto_predict_for_eval=True,
    )
    print(outputs)

    if SYNC_ARTIFACTS_TO_DRIVE:
        local_artifacts = PROJECT_ROOT / 'artifacts'
        COCO_EXPERIMENT_ARTIFACTS.mkdir(parents=True, exist_ok=True)
        shutil.copytree(local_artifacts, COCO_EXPERIMENT_ARTIFACTS, dirs_exist_ok=True)
        print('[OK] Artifacts synced to Drive:', COCO_EXPERIMENT_ARTIFACTS)
else:
    print('Full train+eval skipped')


In [ ]:
# Step X: unified monitoring export (train + val metrics history).
from pathlib import Path
import shutil
import pandas as pd

# This cell builds reusable CSV artifacts for later plots in thesis:
# 1) full epoch history across runs
# 2) last-epoch snapshot per run
# It includes both train-side metrics (losses) and val-side metrics (precision/recall/mAP).

PROJECT_ROOT_RESOLVED = Path(globals().get('PROJECT_ROOT', Path.cwd()))


def _resolve_runtime_cfg_path() -> Path | None:
    candidate_names = ['runtime_cfg_path']
    for name in candidate_names:
        value = globals().get(name)
        if value is not None:
            p = Path(value)
            if p.exists():
                return p

    hints = [
        PROJECT_ROOT_RESOLVED / 'artifacts' / 'dota_runtime_config.yaml',
        PROJECT_ROOT_RESOLVED / 'artifacts' / 'xview_runtime_config.yaml',
        PROJECT_ROOT_RESOLVED / 'artifacts' / 'coco_small_runtime_config.yaml',
        PROJECT_ROOT_RESOLVED / 'artifacts' / 'dota_scaleaware_vs_adaptive_runtime.yaml',
        PROJECT_ROOT_RESOLVED / 'artifacts' / 'runtime_config.yaml',
    ]
    for p in hints:
        if p.exists():
            return p
    return None


def _load_cfg(cfg_path: Path | None):
    if cfg_path is None:
        return None
    try:
        import yaml
        return yaml.safe_load(cfg_path.read_text(encoding='utf-8'))
    except Exception:
        return None


def _resolve_runs_root(cfg: dict | None) -> Path:
    name_candidates = [
        'RUNS_ROOT',
        'RUNS_DRIVE_DIR',
        'DOTA_EXPERIMENT_RUNS',
        'COCO_EXPERIMENT_RUNS',
        'XVIEW_EXPERIMENT_RUNS',
    ]
    for name in name_candidates:
        value = globals().get(name)
        if value is not None:
            p = Path(value)
            if p.exists():
                return p

    if isinstance(cfg, dict):
        p = cfg.get('training', {}).get('project_dir')
        if p:
            rp = Path(p)
            if rp.exists():
                return rp

    fallback = PROJECT_ROOT_RESOLVED / 'runs'
    fallback.mkdir(parents=True, exist_ok=True)
    return fallback


def _resolve_drive_artifacts_root() -> Path | None:
    candidates = [
        globals().get('ARTIFACTS_ROOT'),
        globals().get('DOTA_EXPERIMENT_ARTIFACTS'),
        globals().get('COCO_EXPERIMENT_ARTIFACTS'),
        globals().get('XVIEW_EXPERIMENT_ARTIFACTS'),
        globals().get('ARTIFACTS_DRIVE_DIR'),
        globals().get('DRIVE_ARTIFACTS_ROOT'),
    ]
    for item in candidates:
        if item is None:
            continue
        p = Path(item)
        try:
            p.mkdir(parents=True, exist_ok=True)
            return p
        except Exception:
            continue
    return None


cfg_path = _resolve_runtime_cfg_path()
cfg = _load_cfg(cfg_path)
runs_root = _resolve_runs_root(cfg)

monitor_dir = PROJECT_ROOT_RESOLVED / 'artifacts' / 'monitoring'
monitor_dir.mkdir(parents=True, exist_ok=True)

results_files = sorted(runs_root.rglob('results.csv'))
if not results_files:
    raise FileNotFoundError(f'No results.csv found under runs root: {runs_root}')

frames = []
for csv_path in results_files:
    run_dir = csv_path.parent
    run_name = run_dir.name
    try:
        df = pd.read_csv(csv_path)
    except Exception as exc:
        print(f'[WARN] failed to read {csv_path}: {exc}')
        continue
    if df.empty:
        continue

    if 'epoch' not in df.columns:
        df['epoch'] = list(range(len(df)))

    df['run_name'] = run_name
    df['run_dir'] = run_dir.as_posix()
    frames.append(df)

if not frames:
    raise RuntimeError('No readable non-empty results.csv files were found.')

history_wide = pd.concat(frames, ignore_index=True)

# Keep numeric metric columns + id columns.
id_cols = ['run_name', 'run_dir', 'epoch']
metric_cols = [
    c for c in history_wide.columns
    if c not in id_cols and pd.api.types.is_numeric_dtype(history_wide[c])
]
history_wide = history_wide[id_cols + metric_cols]

history_wide_path = monitor_dir / 'metrics_history_wide.csv'
history_wide.to_csv(history_wide_path, index=False)

# Last-epoch snapshot per run (useful for quick comparison tables).
last_rows = history_wide.sort_values(['run_name', 'epoch']).groupby('run_name', as_index=False).tail(1)
last_rows = last_rows.sort_values('run_name').reset_index(drop=True)
last_rows_path = monitor_dir / 'metrics_last_epoch.csv'
last_rows.to_csv(last_rows_path, index=False)

# Build long-format table for plotting (run/epoch/metric/value).
history_long = history_wide.melt(
    id_vars=['run_name', 'run_dir', 'epoch'],
    value_vars=metric_cols,
    var_name='metric',
    value_name='value',
)

def _metric_split(metric_name: str) -> str:
    m = metric_name.lower()
    if m.startswith('metrics/') or m.startswith('val/'):
        return 'val'
    if m.startswith('train/') or m.endswith('_loss'):
        return 'train'
    if 'loss' in m:
        return 'train'
    return 'other'

history_long['split'] = history_long['metric'].map(_metric_split)
history_long_path = monitor_dir / 'metrics_history_long.csv'
history_long.to_csv(history_long_path, index=False)

# Quick console-friendly view for thesis workflow.
interesting = [
    'train/box_loss', 'train/cls_loss', 'train/dfl_loss',
    'box_loss', 'cls_loss', 'dfl_loss',
    'metrics/precision(B)', 'metrics/recall(B)',
    'metrics/mAP50(B)', 'metrics/mAP50-95(B)',
]
show_cols = ['run_name', 'epoch'] + [c for c in interesting if c in last_rows.columns]
print('[OK] history_wide:', history_wide_path)
print('[OK] history_long:', history_long_path)
print('[OK] last_epoch:', last_rows_path)
print('\nLast-epoch snapshot (train + val):')
print(last_rows[show_cols].to_string(index=False))

# Optional Drive sync for artifacts.
drive_artifacts_root = _resolve_drive_artifacts_root()
if drive_artifacts_root is not None:
    drive_monitor_dir = drive_artifacts_root / 'monitoring'
    drive_monitor_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(history_wide_path, drive_monitor_dir / history_wide_path.name)
    shutil.copy2(history_long_path, drive_monitor_dir / history_long_path.name)
    shutil.copy2(last_rows_path, drive_monitor_dir / last_rows_path.name)
    print('[OK] Monitoring CSVs synced to Drive:', drive_monitor_dir)


In [ ]:
# Step Y: plot preview for train/val metrics (for thesis figures draft).
from pathlib import Path
import shutil
import pandas as pd
import matplotlib.pyplot as plt

RUN_PLOT_PREVIEW = True

if RUN_PLOT_PREVIEW:
    project_root = Path(globals().get('PROJECT_ROOT', Path.cwd()))
    monitor_dir = project_root / 'artifacts' / 'monitoring'
    history_long_path = monitor_dir / 'metrics_history_long.csv'

    if not history_long_path.exists():
        raise FileNotFoundError(
            f'Missing monitoring history: {history_long_path}. '
            'Run the monitoring export cell first.'
        )

    df = pd.read_csv(history_long_path)
    if df.empty:
        raise RuntimeError('metrics_history_long.csv is empty.')

    # Keep numeric rows only.
    df = df[pd.to_numeric(df['value'], errors='coerce').notna()].copy()
    df['value'] = df['value'].astype(float)

    plots_dir = monitor_dir / 'plots'
    plots_dir.mkdir(parents=True, exist_ok=True)

    # Metrics that are usually useful for diploma report.
    metric_candidates = [
        'metrics/mAP50-95(B)',
        'metrics/mAP50(B)',
        'metrics/precision(B)',
        'metrics/recall(B)',
        'train/box_loss',
        'train/cls_loss',
        'train/dfl_loss',
        'box_loss',
        'cls_loss',
        'dfl_loss',
    ]
    metrics = [m for m in metric_candidates if m in set(df['metric'].unique())]

    if not metrics:
        raise RuntimeError('No known metrics found for plotting in metrics_history_long.csv.')

    created = []

    # 1) Per-metric comparison across runs.
    for metric in metrics:
        sub = df[df['metric'] == metric].copy()
        if sub.empty:
            continue

        plt.figure(figsize=(9, 5))
        for run_name, grp in sub.groupby('run_name'):
            grp = grp.sort_values('epoch')
            plt.plot(grp['epoch'], grp['value'], marker='o', markersize=2, linewidth=1.6, label=str(run_name))

        plt.title(f'{metric} vs epoch')
        plt.xlabel('epoch')
        plt.ylabel(metric)
        plt.grid(alpha=0.3)
        plt.legend(loc='best', fontsize=8)
        out = plots_dir / f"metric_{metric.replace('/', '_').replace('(', '').replace(')', '')}.png"
        plt.tight_layout()
        plt.savefig(out, dpi=160)
        plt.close()
        created.append(out)

    # 2) Overview: best val metrics by run (last epoch).
    last_rows = (
        df.sort_values(['run_name', 'epoch'])
          .groupby(['run_name', 'metric'], as_index=False)
          .tail(1)
    )

    val_metrics_for_bar = [m for m in ['metrics/mAP50-95(B)', 'metrics/mAP50(B)', 'metrics/precision(B)', 'metrics/recall(B)'] if m in set(last_rows['metric'])]
    if val_metrics_for_bar:
        pivot = (
            last_rows[last_rows['metric'].isin(val_metrics_for_bar)]
            .pivot_table(index='run_name', columns='metric', values='value', aggfunc='mean')
            .sort_index()
        )

        ax = pivot.plot(kind='bar', figsize=(10, 5), rot=25)
        ax.set_title('Final val metrics by run (last epoch)')
        ax.set_ylabel('value')
        ax.grid(axis='y', alpha=0.3)
        out_bar = plots_dir / 'final_val_metrics_bar.png'
        plt.tight_layout()
        plt.savefig(out_bar, dpi=160)
        plt.close()
        created.append(out_bar)

    print('[OK] Plot files created:')
    for p in created:
        print(' -', p)

    # Optional Drive sync.
    drive_candidates = [
        globals().get('ARTIFACTS_ROOT'),
        globals().get('DOTA_EXPERIMENT_ARTIFACTS'),
        globals().get('COCO_EXPERIMENT_ARTIFACTS'),
        globals().get('XVIEW_EXPERIMENT_ARTIFACTS'),
        globals().get('ARTIFACTS_DRIVE_DIR'),
        globals().get('DRIVE_ARTIFACTS_ROOT'),
    ]
    drive_root = None
    for c in drive_candidates:
        if c is None:
            continue
        try:
            p = Path(c)
            p.mkdir(parents=True, exist_ok=True)
            drive_root = p
            break
        except Exception:
            continue

    if drive_root is not None:
        drive_plots_dir = drive_root / 'monitoring' / 'plots'
        drive_plots_dir.mkdir(parents=True, exist_ok=True)
        for p in created:
            shutil.copy2(p, drive_plots_dir / p.name)
        print('[OK] Plot files synced to Drive:', drive_plots_dir)
else:
    print('Plot preview skipped. Set RUN_PLOT_PREVIEW=True to enable.')
